# Chapter 14: Fine-tuning & Adaptation Techniques
## Notebook 02 — Parameter-Efficient Fine-tuning (PEFT) and LoRA

Full fine-tuning updates **every parameter** in a billion-scale model. That's expensive in memory, slow, and produces one giant artifact per task. **Parameter-efficient fine-tuning (PEFT)** trains only a tiny fraction of the weights — typically **less than 1%** — while matching full-FT quality on most tasks.

We focus on **LoRA** (Hu et al., 2021), the most widely deployed PEFT method, and implement it from scratch in NumPy. We then survey the broader PEFT family (QLoRA, adapters, prefix tuning, IA³) and discuss merging adapters and serving multiple at once.

### What you'll learn

| Topic | Section |
|-------|--------|
| Full FT vs PEFT trade-offs | §2 |
| LoRA math and a NumPy implementation | §3 |
| Train a tiny LoRA adapter on a regression toy | §4 |
| QLoRA conceptual (4-bit base + LoRA) | §5 |
| Adapters, prefix tuning, IA³ | §6 |
| Merging adapters and multi-adapter serving | §7 |
| Sketch of the `peft` library | §8 |

**Estimated time:** 2.5 hours

---
*Generated by Berta AI*

---
## 1. Setup

In [ ]:
import os, sys, math
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import config
from peft_utils import (
    LinearLayer, LoRALayer, apply_lora_to, merge_lora,
    count_trainable_params, AdapterRegistry,
)

np.random.seed(42)
print('LoRA defaults:  rank =', config.LORA_RANK, '  alpha =', config.LORA_ALPHA)

---
## 2. Full Fine-tuning vs PEFT: the Trade-offs

For a model with `D` parameters and a hidden size `d`, a LoRA adapter on one linear layer has only `2 * r * d` parameters where `r` (rank) is typically 4–64. Across a 7B model, that's often a few million trainable parameters total — **~0.1% of the model**.

| Property | Full FT | PEFT (LoRA) |
|---|---|---|
| Trainable params | 100% | ~0.05–1% |
| Optimizer state | huge (Adam = 2× model) | tiny |
| Storage per task | full checkpoint | small adapter (MBs) |
| Multi-task serving | one model per task | many adapters, one base |
| Quality ceiling | highest | usually within 1% on most tasks |
| Catastrophic forgetting | high | low (frozen base) |

In [ ]:
# Visualize parameter counts for a single linear layer (in_dim x out_dim).
in_dim, out_dim = 4096, 4096
ranks = [2, 4, 8, 16, 32, 64]
full = in_dim * out_dim
lora = [r * (in_dim + out_dim) for r in ranks]
df = pd.DataFrame({'rank': ranks, 'lora_params': lora, 'full_ft_params': full,
                   'fraction_of_full': [l / full for l in lora]})
df

In [ ]:
plt.bar([str(r) for r in ranks], [l / full * 100 for l in lora])
plt.ylabel('LoRA params as % of full FT')
plt.xlabel('LoRA rank r')
plt.title(f'A single {in_dim}x{out_dim} linear layer')
plt.show()

---
## 3. LoRA: the Math

Take a frozen linear layer with weights `W` of shape `(out, in)`. LoRA learns a **low-rank update** $\Delta W = B A$ where:

- $A \in \mathbb{R}^{r \times in}$  (small, initialized random)
- $B \in \mathbb{R}^{out \times r}$  (small, initialized to zero)
- $r \ll \min(in, out)$ is the **rank**.

The forward pass becomes:

$$ y = x W^\top + x A^\top B^\top \cdot \frac{\alpha}{r} $$

where $\alpha$ is a scaling hyperparameter that decouples the update magnitude from the rank. Because **B starts at zero**, the adapter is a no-op at initialization — training begins from the base model's behavior.

In [ ]:
# Construct a frozen base layer and a LoRA adapter on top.
base = LinearLayer.random(in_features=64, out_features=32, seed=1)
lora = apply_lora_to(base, rank=8, alpha=16.0)

x = np.random.default_rng(0).normal(size=(5, 64))
out_no_adapter = base.forward(x)
out_with_adapter = lora.forward(x, base)
print('Same outputs at init?', np.allclose(out_no_adapter, out_with_adapter))

params = count_trainable_params(base, lora)
print(params)
print(f'Trainable fraction: {params["trainable_fraction"] * 100:.2f}%')

---
## 4. Train a Tiny LoRA Adapter

We freeze a randomly-initialized base layer and ask LoRA to adapt it to a **synthetic regression target** that the base layer cannot solve on its own. We compare against (a) the frozen base (no training) and (b) full fine-tuning of the base.

In [ ]:
rng = np.random.default_rng(7)
in_dim, out_dim, n = 32, 8, 200
X = rng.normal(size=(n, in_dim)).astype(np.float64)
true_W = rng.normal(scale=0.3, size=(out_dim, in_dim))
Y = X @ true_W.T + 0.05 * rng.normal(size=(n, out_dim))

split = int(0.8 * n)
X_tr, X_va, Y_tr, Y_va = X[:split], X[split:], Y[:split], Y[split:]

base = LinearLayer.random(in_features=in_dim, out_features=out_dim, seed=3)
lora = apply_lora_to(base, rank=4, alpha=8.0, seed=5)

def mse(y, yhat):
    return float(((y - yhat) ** 2).mean())

frozen_mse = mse(Y_va, base.forward(X_va))
print(f'Frozen base val MSE: {frozen_mse:.4f}')

In [ ]:
# Manual gradient descent on A and B only (base is frozen).
lr = 0.01
history = {'lora_train': [], 'lora_val': [], 'full_train': [], 'full_val': []}
for step in range(200):
    yhat = lora.forward(X_tr, base)
    err = (yhat - Y_tr) / X_tr.shape[0]
    # delta = (X @ A.T) @ B.T * scaling
    s = lora.scaling
    XA = X_tr @ lora.A.T            # (n, r)
    grad_B = (err.T @ XA) * s        # (out, r)
    grad_A = (err @ lora.B * s).T @ X_tr  # (r, in)
    lora.B -= lr * grad_B
    lora.A -= lr * grad_A
    history['lora_train'].append(mse(Y_tr, lora.forward(X_tr, base)))
    history['lora_val'].append(mse(Y_va, lora.forward(X_va, base)))

# Compare to full fine-tune of W (least squares, the optimum).
W_full, *_ = np.linalg.lstsq(X_tr, Y_tr, rcond=None)
full_val = mse(Y_va, X_va @ W_full)

plt.plot(history['lora_train'], label='LoRA train')
plt.plot(history['lora_val'], label='LoRA val')
plt.axhline(frozen_mse, color='gray', linestyle='--', label='frozen base')
plt.axhline(full_val, color='green', linestyle=':', label='full FT (LS)')
plt.xlabel('step'); plt.ylabel('MSE'); plt.legend(); plt.title('LoRA vs full FT vs frozen base')
plt.show()
print(f'LoRA val MSE: {history["lora_val"][-1]:.4f}  |  full FT val MSE: {full_val:.4f}  |  frozen: {frozen_mse:.4f}')
print(f'LoRA trains {lora.A.size + lora.B.size} params vs full FT {base.W.size} ({(lora.A.size + lora.B.size) / base.W.size * 100:.1f}%)')

**Reading the curves**: LoRA at rank 4 closes most of the gap between the frozen base and full fine-tuning while training a small fraction of the parameters. Increase the rank (try 8, 16) and you usually close the rest of the gap, at proportional cost.

---
## 5. QLoRA: 4-bit Base + LoRA

**QLoRA** (Dettmers et al., 2023) shrinks memory further by storing the *frozen* base in 4-bit precision (NF4) while keeping LoRA adapters in 16-bit. Forward passes dequantize on the fly. The adapter still updates in fp16/bf16, so quality matches LoRA on top of fp16 in most cases.

Effect:

- 7B model in 4-bit: ~4 GB GPU memory for the base.
- LoRA optimizer state: tens of MB.
- A 7B model fine-tunes on a single 24 GB consumer GPU.

We **don't** quantize anything in this notebook (`bitsandbytes` requires a GPU), but the conceptual recipe is:

1. Load base in 4-bit (NF4 + double quantization).
2. Wrap target modules with LoRA in fp16/bf16.
3. Train normally; the base weights never see a gradient.

---
## 6. Other PEFT Methods

**Adapters** (Houlsby et al., 2019): insert small bottleneck MLPs between transformer sub-layers. Trains ~1% of parameters; adds inference cost (extra MLPs in the forward pass).

**Prefix tuning** (Li & Liang, 2021): prepend a small set of *trainable* key/value vectors at every attention layer. The base weights are frozen; only the prefixes update. Very few parameters; tricky to tune for shorter sequences.

**IA³** (Liu et al., 2022): rescales activations with learned vectors (one per attention head and feed-forward block). Even smaller than LoRA — typically <0.01% of parameters — but lower ceiling on hard tasks.

Choose by (a) parameter budget, (b) inference latency target, (c) task difficulty:

| Method | Trainable | Inference cost | Quality ceiling |
|---|---|---|---|
| LoRA | ~0.1–1% | low (mergeable) | high |
| QLoRA | same | low | same as LoRA |
| Adapters | ~1% | small extra MLPs | high |
| Prefix tuning | tiny | small extra KV | medium |
| IA³ | tiniest | negligible | medium |

---
## 7. Merging Adapters and Multi-Adapter Serving

A LoRA update `B A * (alpha / r)` is a matrix the same shape as `W`. We can **merge** it into the base for zero-overhead inference, or keep it separate to **swap adapters per request** (multi-tenancy).

Below we demonstrate both: merge produces identical outputs, and the registry serves several adapters from one frozen base.

In [ ]:
rng = np.random.default_rng(11)
base = LinearLayer.random(in_features=16, out_features=8, seed=1)
lora = apply_lora_to(base, rank=4, alpha=8.0)
lora.B = rng.normal(scale=0.05, size=lora.B.shape)
x = rng.normal(size=(3, 16))

out_lora = lora.forward(x, base)
merged = merge_lora(base, lora)
out_merged = merged.forward(x)
print('Merge equivalence:', np.allclose(out_lora, out_merged))
print('Merged W is the same shape as base W:', merged.W.shape == base.W.shape)

In [ ]:
# Multi-adapter serving — same base, different adapters chosen per request.
registry = AdapterRegistry(base=base)
registry.add('customer_support', apply_lora_to(base, rank=4, alpha=8.0, seed=10))
registry.add('legal_summary',    apply_lora_to(base, rank=4, alpha=8.0, seed=20))
registry.add('code_review',      apply_lora_to(base, rank=4, alpha=8.0, seed=30))
print('Registered adapters:', registry.list())

for name in registry.list():
    y = registry.forward(x, name)
    print(f'  adapter={name:>18}  output mean={y.mean():+.4f}  std={y.std():.4f}')

**Why this matters in production**: one frozen 7B base in GPU memory + several tiny adapters lets you serve dozens of fine-tuned variants without paying for dozens of full models. Frameworks like vLLM, TGI, and LoRAX exploit exactly this.

---
## 8. Sketch: the `peft` Library

On real models, you don't write LoRA by hand — you use Hugging Face `peft`. The cell below stays informative even when `peft` isn't installed.

In [ ]:
try:
    from peft import LoraConfig, get_peft_model, TaskType
    print('peft is installed. Sketch:')
    sketch = '''
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

base = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(base, config)
model.print_trainable_parameters()
# -> trainable params: 1.1M  ||  all params: 1.24B  ||  trainable%: 0.09

# train as usual, then save just the adapter:
model.save_pretrained("adapters/lora_v1")
# at inference: PeftModel.from_pretrained(base, "adapters/lora_v1")
'''
    print(sketch)
except ImportError:
    print('peft not installed. Conceptual flow:')
    print('  1) load base model from transformers')
    print('  2) wrap with LoraConfig (rank, alpha, target_modules)')
    print('  3) train; save only the adapter (small file)')
    print('  4) at serve time, PeftModel.from_pretrained(base, adapter_dir)')

---
## 9. Key Takeaways

- **LoRA** trains <1% of the parameters by learning a low-rank update `B A`. B starts at zero, so init is a no-op.
- **Rank** controls capacity; **alpha/r** controls magnitude. Defaults: r=8, alpha=16.
- **Merging** folds the adapter into the base for zero-overhead inference; **registries** keep them separate for multi-tenancy.
- **QLoRA** = 4-bit base + LoRA adapters; fine-tunes 7B on consumer GPUs.
- **Adapters / prefix / IA³** are alternatives with different trade-offs; LoRA is the strong default.

Next — **Notebook 03**: instruction tuning at scale, preference data, DPO, evaluation, and deployment.

---
*Generated by Berta AI*